# Construcción del dataset de modelado: `mag_14(t+1)`

Dataset final para predecir la concentración de `mag_14` del día siguiente.

Decisiones:
- Solo estaciones de aire que miden `mag_14`.
- `mag_8` y `mag_14` como variables de contaminación.
- Lags de aire: 0, 1, 2, 3, 7 y 14.
- Rolling de `mag_14`: medias de 3, 7 y 14 días y desviación estándar de 7 días.
- Meteorología exclusivamente desde `enriched.meteo_final`.
- La estación meteorológica es la más cercana entre las estaciones presentes en `enriched.meteo_final`.
- La disponibilidad de variables meteorológicas se determina por los pares existentes en `enriched.meteo_final`.
- Lags meteorológicos de 0 a 6 días.
- Target: `mag_14(t+1)`.
- Los `NaN` meteorológicos estructurales se conservan.


In [98]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

# Localizar raíz del proyecto
ROOT = Path.cwd()

while not (ROOT / "src" / "database" / "TFM.duckdb").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError(
            "No se ha encontrado la raíz del proyecto."
        )
    ROOT = ROOT.parent

print(f"Raíz del proyecto: {ROOT}")


Raíz del proyecto: c:\Users\dasab\Desktop\MASTER\TFM


In [99]:
DB_PATH = ROOT / "src" / "database" / "TFM.duckdb"

con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

print(f"Conectado a: {DB_PATH}")


Conectado a: c:\Users\dasab\Desktop\MASTER\TFM\src\database\TFM.duckdb


In [100]:
TARGET_MAG = 14

AIR_LAGS = [0, 1, 2, 3, 7, 14]
METEO_LAGS = [0, 1, 2, 3, 4, 5, 6]


## 1. Estaciones de aire objetivo

In [101]:
stations = con.execute("""
    SELECT DISTINCT estacion
    FROM enriched.calidad_aire_final
    WHERE magnitud = 14
    ORDER BY estacion
""").df()

target_stations = stations["estacion"].tolist()

print("Estaciones objetivo:", target_stations)
print("Número:", len(target_stations))


Estaciones objetivo: [8, 16, 17, 18, 24, 27, 35, 39, 49, 54, 58, 59, 60]
Número: 13


## 2. Aire: magnitudes 8 y 14

In [102]:
air = con.execute("""
    SELECT
        estacion,
        fecha,
        magnitud,
        uom_value
    FROM enriched.calidad_aire_final
    WHERE magnitud IN (8, 14)
      AND estacion IN (
          SELECT DISTINCT estacion
          FROM enriched.calidad_aire_final
          WHERE magnitud = 14
      )
    ORDER BY estacion, fecha
""").df()

air["fecha"] = pd.to_datetime(air["fecha"])

air = (
    air
    .pivot_table(
        index=["estacion", "fecha"],
        columns="magnitud",
        values="uom_value",
        aggfunc="first"
    )
    .reset_index()
    .rename(columns={8: "mag_8", 14: "mag_14"})
)

print("Shape aire:", air.shape)


Shape aire: (23751, 4)


## 3. Lags de contaminación

In [103]:
air = air.sort_values(["estacion", "fecha"]).reset_index(drop=True)

for col in ["mag_8", "mag_14"]:
    for lag in AIR_LAGS:
        air[f"{col}_lag_{lag}"] = (
            air.groupby("estacion")[col].shift(lag)
        )


## 4. Rolling de `mag_14`

In [104]:
air["mag_14_mean_3"] = (
    air.groupby("estacion")["mag_14"]
    .transform(lambda x: x.rolling(3, min_periods=3).mean())
)

air["mag_14_mean_7"] = (
    air.groupby("estacion")["mag_14"]
    .transform(lambda x: x.rolling(7, min_periods=7).mean())
)

air["mag_14_mean_14"] = (
    air.groupby("estacion")["mag_14"]
    .transform(lambda x: x.rolling(14, min_periods=14).mean())
)

air["mag_14_std_7"] = (
    air.groupby("estacion")["mag_14"]
    .transform(lambda x: x.rolling(7, min_periods=7).std())
)


## 5. Target: `mag_14` del día siguiente

In [105]:
air["target_mag_14"] = (
    air.groupby("estacion")["mag_14"].shift(-1)
)


## 6. Variables temporales

In [106]:
air["day_of_week"] = air["fecha"].dt.dayofweek
air["month"] = air["fecha"].dt.month
air["day_of_year"] = air["fecha"].dt.dayofyear

air["dow_sin"] = np.sin(2 * np.pi * air["day_of_week"] / 7)
air["dow_cos"] = np.cos(2 * np.pi * air["day_of_week"] / 7)

air["doy_sin"] = np.sin(2 * np.pi * air["day_of_year"] / 365.25)
air["doy_cos"] = np.cos(2 * np.pi * air["day_of_year"] / 365.25)

laborable = con.execute("""
    SELECT DISTINCT
        fecha,
        es_laborable_madrid_ciudad
    FROM enriched.calidad_aire_final
""").df()

laborable["fecha"] = pd.to_datetime(laborable["fecha"])

air = air.merge(
    laborable,
    on="fecha",
    how="left"
)


## 7. Variables meteorológicas disponibles

In [107]:
METEO_VARIABLES = con.execute("""
    SELECT DISTINCT
        variable
    FROM enriched.meteo_final
    WHERE variable <> 'insolacion'
    ORDER BY variable
""").df()["variable"].tolist()

print("Variables meteorológicas utilizadas:")
for variable in METEO_VARIABLES:
    print(f" - {variable}")

Variables meteorológicas utilizadas:
 - direccion_racha_max
 - humedad_max
 - humedad_media
 - humedad_min
 - precipitacion
 - presion_max
 - presion_min
 - temp_max
 - temp_media
 - temp_min
 - viento_racha
 - viento_velocidad


## 8. Mapping aire → estación meteorológica más cercana válida

In [108]:
mapping = con.execute("""
    WITH estaciones_14 AS (
        SELECT DISTINCT estacion
        FROM enriched.calidad_aire_final
        WHERE magnitud = 14
    ),

    estaciones_meteo_validas AS (
        SELECT DISTINCT estacion_id
        FROM enriched.meteo_final
    ),

    distancias AS (
        SELECT
            e.ESTACION AS estacion,
            k AS estacion_meteo,
            CAST(
                json_extract(
                    e.distancias_meteo,
                    '$."' || k || '"'
                ) AS DOUBLE
            ) AS distancia_meteo
        FROM enriched.dim_estaciones_aire e
        CROSS JOIN LATERAL unnest(
            json_keys(e.distancias_meteo)
        ) AS t(k)
        WHERE e.ESTACION IN (
            SELECT estacion
            FROM estaciones_14
        )
        AND k IN (
            SELECT estacion_id
            FROM estaciones_meteo_validas
        )
    ),

    ranking AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY estacion
                ORDER BY distancia_meteo
            ) AS rn
        FROM distancias
    )

    SELECT
        estacion,
        estacion_meteo,
        distancia_meteo
    FROM ranking
    WHERE rn = 1
    ORDER BY estacion
""").df()

print(mapping.to_string(index=False))


 estacion estacion_meteo  distancia_meteo
        8           3195             1.20
       16           3195             4.56
       17           3200             5.48
       18           3195             4.95
       24           3196             5.85
       27           3129             1.87
       35           3195             2.33
       39           3195             7.97
       49           3195             0.68
       54           3195             7.04
       58           3195            14.41
       59           3129             4.52
       60           3195             9.97


## 9. Cargar meteorología desde `enriched.meteo_final`

In [109]:
meteo = con.execute("""
    SELECT
        estacion_id,
        fecha,
        variable,
        uom_value
    FROM enriched.meteo_final
""").df()

meteo["fecha"] = pd.to_datetime(meteo["fecha"])

print("Filas meteorológicas:", len(meteo))


Filas meteorológicas: 89523


## 10. Aplicar mapping

In [110]:
meteo = meteo.merge(
    mapping,
    left_on="estacion_id",
    right_on="estacion_meteo",
    how="inner"
)

print(
    "Estaciones de aire con meteorología:",
    meteo["estacion"].nunique()
)


Estaciones de aire con meteorología: 13


## 11. Pivot meteorológico

In [111]:
meteo = (
    meteo
    .pivot_table(
        index=["estacion", "fecha"],
        columns="variable",
        values="uom_value",
        aggfunc="first"
    )
    .reset_index()
    .sort_values(["estacion", "fecha"])
)


## 12. Lags meteorológicos

In [112]:
for col in METEO_VARIABLES:
    for lag in METEO_LAGS:
        meteo[f"{col}_lag_{lag}"] = (
            meteo.groupby("estacion")[col].shift(lag)
        )


## 13. Unir aire + meteorología

In [113]:
model_df = air.merge(
    meteo,
    on=["estacion", "fecha"],
    how="left"
)

print("Dataset combinado:", model_df.shape)


Dataset combinado: (23751, 126)


## 14. Selección explícita de features

In [114]:
air_feature_cols = []

for col in ["mag_8", "mag_14"]:
    air_feature_cols += [
        f"{col}_lag_{lag}"
        for lag in AIR_LAGS
    ]

air_feature_cols += [
    "mag_14_mean_3",
    "mag_14_mean_7",
    "mag_14_mean_14",
    "mag_14_std_7",
]

meteo_feature_cols = [
    f"{col}_lag_{lag}"
    for col in METEO_VARIABLES
    for lag in METEO_LAGS
]

temporal_cols = [
    "day_of_week",
    "month",
    "day_of_year",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos",
    "es_laborable_madrid_ciudad",
]

final_cols = (
    ["estacion", "fecha", "target_mag_14"]
    + air_feature_cols
    + meteo_feature_cols
    + temporal_cols
)

model_df = model_df[final_cols].copy()

print("Número de features:", len(final_cols) - 3)


Número de features: 108


## 15. Eliminar solo filas sin historial suficiente o sin target

In [115]:
required_history = [
    "mag_14_lag_14",
    "mag_8_lag_14",
    "mag_14_mean_14",
    "mag_14_std_7",
    "target_mag_14",
]

before = len(model_df)

model_df = model_df.dropna(
    subset=required_history
).copy()

after = len(model_df)

print(f"Filas antes:   {before}")
print(f"Filas después: {after}")
print(f"Eliminadas:    {before - after}")


Filas antes:   23751
Filas después: 23556
Eliminadas:    195


## 16. Comprobaciones finales

In [116]:
print("Shape:", model_df.shape)
print("Estaciones:", model_df["estacion"].nunique())
print("Lista:", sorted(model_df["estacion"].unique()))
print("Fecha:", model_df["fecha"].min(), "→", model_df["fecha"].max())


Shape: (23556, 111)
Estaciones: 13
Lista: [np.int64(8), np.int64(16), np.int64(17), np.int64(18), np.int64(24), np.int64(27), np.int64(35), np.int64(39), np.int64(49), np.int64(54), np.int64(58), np.int64(59), np.int64(60)]
Fecha: 2020-01-15 00:00:00 → 2024-12-30 00:00:00


In [117]:
missing = (
    model_df
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(missing.head(30).to_string())


humedad_max_lag_6    7.692308
humedad_max_lag_5    7.692308
humedad_min_lag_5    7.692308
humedad_min_lag_3    7.692308
humedad_min_lag_2    7.692308
humedad_min_lag_1    7.692308
humedad_min_lag_0    7.692308
humedad_min_lag_6    7.692308
humedad_max_lag_3    7.692308
humedad_min_lag_4    7.692308
humedad_max_lag_0    7.692308
humedad_max_lag_4    7.692308
humedad_max_lag_1    7.692308
humedad_max_lag_2    7.692308
estacion             0.000000
mag_14_lag_14        0.000000
mag_14_lag_7         0.000000
mag_14_lag_3         0.000000
mag_14_lag_2         0.000000
mag_14_lag_1         0.000000
mag_14_lag_0         0.000000
mag_8_lag_14         0.000000
mag_8_lag_7          0.000000
mag_8_lag_3          0.000000
mag_8_lag_2          0.000000
mag_8_lag_1          0.000000
mag_8_lag_0          0.000000
target_mag_14        0.000000
fecha                0.000000
mag_14_mean_14       0.000000


In [118]:
assert model_df["target_mag_14"].isna().sum() == 0
assert model_df["mag_14_lag_0"].isna().sum() == 0
assert model_df["mag_8_lag_0"].isna().sum() == 0

assert not any(
    "mag_9" in c or "mag_10" in c
    for c in model_df.columns
)

print("Comprobaciones principales OK")


Comprobaciones principales OK


## 17. Guardar dataset final

In [119]:
OUTPUT_PATH = (
    ROOT
    / "data"
    / "modeling"
    / "dataset_mag14_t1.parquet"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

model_df.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(f"Dataset guardado: {OUTPUT_PATH}")


Dataset guardado: c:\Users\dasab\Desktop\MASTER\TFM\data\modeling\dataset_mag14_t1.parquet


## 18. Comprobar el parquet guardado

In [120]:
df_check = pd.read_parquet(OUTPUT_PATH)

print("Shape guardado:", df_check.shape)
print("Columnas:", len(df_check.columns))
print(
    "Fecha:",
    df_check["fecha"].min(),
    "→",
    df_check["fecha"].max()
)


Shape guardado: (23556, 111)
Columnas: 111
Fecha: 2020-01-15 00:00:00 → 2024-12-30 00:00:00


## 19. Cerrar DuckDB

In [121]:
con.close()
print("Conexión DuckDB cerrada.")


Conexión DuckDB cerrada.
